# Master Results Table — Cross-Technique Comparison

**Purpose.** Single publication-ready comparison of every modelling technique across
all targets (BMWP, Perlidae, Helicopsychidae), assembled **only by loading the
metric CSVs** produced by the model notebooks (no modelling code here).
**Supports article section:** Results — Table 9.

Sources (run those notebooks first): `01f_fuzzy_final` →
`metrics_fuzzy_redesigned.csv`, `spearman_comparison.csv` & `numerical_bmwp_fuzzy.csv`;
`02_logistic_regression` → `metrics_logistic.csv`;
`03b_classification_trees_LOOCV` → `metrics_trees.csv`;
`04_negative_binomial_regression` → `metrics_nb.csv` & `numerical_bmwp_nbr.csv`;
`05_bmwp_spearman_validation` → `numerical_bmwp_insample.csv`.

In [1]:
# Load all metric CSVs produced by the model notebooks (this notebook runs in outputs/)
import pandas as pd
import numpy as np
from IPython.display import Markdown, display

OUT = ""   # this notebook lives in outputs/, CSVs are alongside it

def load(name):
    return pd.read_csv(OUT + name)

fuzzy = load("metrics_fuzzy_redesigned.csv")
logistic = load("metrics_logistic.csv")
trees = load("metrics_trees.csv")
nb = load("metrics_nb.csv")
spear = load("spearman_comparison.csv")

# Original fuzzy in-sample reference rows (n=14; from 01d/01e, leakage present)
insample = pd.DataFrame([
    {'target': 'BMWP', 'technique': 'Fuzzy logic (original)', 'validation': 'In-sample*', 'n': 14,
     'precision': 0.778, 'recall': 0.800, 'f1': 0.788, 'accuracy': 0.929, 'kappa': 0.878},
    {'target': 'Perlidae', 'technique': 'Fuzzy logic (original)', 'validation': 'In-sample*', 'n': 14,
     'precision': 0.929, 'recall': 0.938, 'f1': 0.928, 'accuracy': 0.929, 'kappa': 0.857},
    {'target': 'Helicopsychidae', 'technique': 'Fuzzy logic (original)', 'validation': 'In-sample*', 'n': 14,
     'precision': 0.875, 'recall': 0.955, 'f1': 0.905, 'accuracy': 0.929, 'kappa': 0.811},
])
allm = pd.concat([fuzzy, logistic, trees, nb, insample], ignore_index=True)
allm[['precision', 'recall', 'f1', 'accuracy', 'kappa']] = \
    allm[['precision', 'recall', 'f1', 'accuracy', 'kappa']].astype(float).round(3)
allm

,target,technique,validation,n,precision,recall,f1,accuracy,kappa
0,BMWP,Fuzzy logic (redesigned),LOOCV,18,0.310,0.317,0.308,0.556,0.345
1,Perlidae,Fuzzy logic (redesigned),LOOCV,18,0.833,0.833,0.800,0.778,0.586
2,Helicopsychidae,Fuzzy logic (redesigned),LOOCV,18,0.617,0.633,0.582,0.611,0.192
3,Perlidae,Logistic regression,LOOCV,18,0.731,0.708,0.610,0.611,0.323
4,Helicopsychidae,Logistic regression,LOOCV,18,0.615,0.667,0.438,0.444,0.143
5,Perlidae,Classification trees,LOOCV,18,0.685,0.667,0.673,0.722,0.348
6,Helicopsychidae,Classification trees,LOOCV,18,0.688,0.633,0.652,0.833,0.308
7,BMWP,Negative binomial reg.,LOOCV,18,0.149,0.200,0.171,0.389,0.062
8,BMWP,Fuzzy logic (original),In-sample*,14,0.778,0.800,0.788,0.929,0.878
9,Perlidae,Fuzzy logic (original),In-sample*,14,0.929,0.938,0.928,0.929,0.857


In [2]:
# Helper: render a markdown table, bolding the best value per metric among out-of-sample rows
METRICS = ['precision', 'recall', 'f1', 'accuracy', 'kappa']
def render(df, title):
    df = df.copy().reset_index(drop=True)
    oos = df['validation'].str.startswith('In-sample') == False
    best = {m: df.loc[oos, m].max() for m in METRICS}
    header = '| Technique | Validation | n | Precision | Recall | F1 | Accuracy | Kappa |'
    sep = '|---|---|---|---|---|---|---|---|'
    lines = [f'### {title}', '', header, sep]
    for i, r in df.iterrows():
        cells_ = [r['technique'], r['validation'], str(int(r['n']))]
        for m in METRICS:
            v = f"{r[m]:.3f}"
            if oos.loc[i] and r[m] == best[m]:
                v = f"**{v}**"
            cells_.append(v)
        lines.append('| ' + ' | '.join(cells_) + ' |')
    lines += ['', '\\*In-sample: rules derived from the same observations used for evaluation (data leakage).']
    display(Markdown('\n'.join(lines)))

## Table A — Habitat suitability models (Perlidae and Helicopsychidae)

In [3]:
# Table A: per-taxon habitat models, ordered by technique
order = ['Fuzzy logic (redesigned)', 'Logistic regression', 'Classification trees', 'Fuzzy logic (original)']
def ordered(df):
    df = df.copy()
    df['__o'] = df['technique'].map({t: i for i, t in enumerate(order)})
    return df.sort_values('__o').drop(columns='__o')

tableA_rows = []
for taxon in ['Perlidae', 'Helicopsychidae']:
    sub = ordered(allm[allm['target'] == taxon])
    render(sub, f"Table A — {taxon}")
    tableA_rows.append(sub)

tableA = pd.concat(tableA_rows, ignore_index=True)
tableA = tableA[['target', 'technique', 'validation', 'n', 'precision', 'recall', 'f1', 'accuracy', 'kappa']]
tableA.to_csv(OUT + "table_A_habitat.csv", index=False)
print("Saved outputs/table_A_habitat.csv")

### Table A — Perlidae

| Technique | Validation | n | Precision | Recall | F1 | Accuracy | Kappa |
|---|---|---|---|---|---|---|---|
| Fuzzy logic (redesigned) | LOOCV | 18 | **0.833** | **0.833** | **0.800** | **0.778** | **0.586** |
| Logistic regression | LOOCV | 18 | 0.731 | 0.708 | 0.610 | 0.611 | 0.323 |
| Classification trees | LOOCV | 18 | 0.685 | 0.667 | 0.673 | 0.722 | 0.348 |
| Fuzzy logic (original) | In-sample* | 14 | 0.929 | 0.938 | 0.928 | 0.929 | 0.857 |

\*In-sample: rules derived from the same observations used for evaluation (data leakage).

### Table A — Helicopsychidae

| Technique | Validation | n | Precision | Recall | F1 | Accuracy | Kappa |
|---|---|---|---|---|---|---|---|
| Fuzzy logic (redesigned) | LOOCV | 18 | 0.617 | 0.633 | 0.582 | 0.611 | 0.192 |
| Logistic regression | LOOCV | 18 | 0.615 | **0.667** | 0.438 | 0.444 | 0.143 |
| Classification trees | LOOCV | 18 | **0.688** | 0.633 | **0.652** | **0.833** | **0.308** |
| Fuzzy logic (original) | In-sample* | 14 | 0.875 | 0.955 | 0.905 | 0.929 | 0.811 |

\*In-sample: rules derived from the same observations used for evaluation (data leakage).

Saved outputs/table_A_habitat.csv


## Table B — Water quality index model (BMWP/Col)

In [4]:
# Table B: BMWP class-level models, ordered by technique
orderB = ['Fuzzy logic (redesigned)', 'Negative binomial reg.', 'Fuzzy logic (original)']
subB = allm[allm['target'] == 'BMWP'].copy()
subB['__o'] = subB['technique'].map({t: i for i, t in enumerate(orderB)})
subB = subB.sort_values('__o').drop(columns='__o')
render(subB, "Table B — BMWP/Col")
tableB = subB[['target', 'technique', 'validation', 'n', 'precision', 'recall', 'f1', 'accuracy', 'kappa']]
tableB.to_csv(OUT + "table_B_bmwp.csv", index=False)
print("Saved outputs/table_B_bmwp.csv")

### Table B — BMWP/Col

| Technique | Validation | n | Precision | Recall | F1 | Accuracy | Kappa |
|---|---|---|---|---|---|---|---|
| Fuzzy logic (redesigned) | LOOCV | 18 | **0.310** | **0.317** | **0.308** | **0.556** | **0.345** |
| Negative binomial reg. | LOOCV | 18 | 0.149 | 0.200 | 0.171 | 0.389 | 0.062 |
| Fuzzy logic (original) | In-sample* | 14 | 0.778 | 0.800 | 0.788 | 0.929 | 0.878 |

\*In-sample: rules derived from the same observations used for evaluation (data leakage).

Saved outputs/table_B_bmwp.csv


## Table C — Numerical performance (BMWP models)

Numerical accuracy of the continuous/crisp BMWP predictions on the 0–120 BMWP/Col
scale (MAE, RMSE, R²) together with Spearman's rank correlation. Loaded from the
numerical CSVs produced by `01f`, `04` and `05`. The redesigned fuzzy LOOCV row uses
n = 17 (one coverage failure excluded from numerical metrics).

In [5]:
# Table C: numerical BMWP performance from the three numerical CSVs
num_fuzzy = load("numerical_bmwp_fuzzy.csv")        # Fuzzy redesigned, LOOCV, n=17
num_nbr = load("numerical_bmwp_nbr.csv")            # NBR, LOOCV, n=18
num_insample = load("numerical_bmwp_insample.csv")  # Fuzzy original, In-sample, n=14
tableC = pd.concat([num_fuzzy, num_nbr, num_insample], ignore_index=True)
tableC = tableC[['model', 'validation', 'n', 'mae', 'rmse', 'r2', 'rs', 'p_value']]
tableC.to_csv(OUT + "table_C_numerical_bmwp.csv", index=False)

# Render as a markdown table
header = '| Model | Validation | n | MAE | RMSE | R² | Spearman rs | p-value |'
sep = '|---|---|---|---|---|---|---|---|'
lines = ['### Table C — Numerical performance (BMWP)', '', header, sep]
for _, r in tableC.iterrows():
    lines.append(f"| {r['model']} | {r['validation']} | {int(r['n'])} | {r['mae']:.2f} | "
                 f"{r['rmse']:.2f} | {r['r2']:.3f} | {r['rs']:.3f} | {float(r['p_value']):.6f} |")
lines += ['', '*Fuzzy redesigned uses 17 successful folds (1 coverage failure excluded).*']
display(Markdown('\n'.join(lines)))
print("Saved outputs/table_C_numerical_bmwp.csv")

### Table C — Numerical performance (BMWP)

| Model | Validation | n | MAE | RMSE | R² | Spearman rs | p-value |
|---|---|---|---|---|---|---|---|
| Fuzzy redesigned | LOOCV | 17 | 21.27 | 26.19 | 0.324 | 0.533 | 0.027400 |
| NBR | LOOCV | 18 | 27.43 | 30.18 | 0.063 | 0.344 | 0.162600 |
| Fuzzy original | In-sample | 14 | 9.83 | 12.29 | 0.842 | 0.827 | 0.000265 |

*Fuzzy redesigned uses 17 successful folds (1 coverage failure excluded).*

Saved outputs/table_C_numerical_bmwp.csv


## Spearman correlation (BMWP only)

In [6]:
# Spearman summary table (in-sample vs LOOCV)
sp = spear.copy()
sp['rs'] = sp['rs'].astype(float).round(3)
lines = ['| System | Evaluation | n | rs | p-value |', '|---|---|---|---|---|']
for _, r in sp.iterrows():
    lines.append(f"| {r['system']} | {r['evaluation']} | {int(r['n'])} | {r['rs']:.3f} | {float(r['p_value']):.6f} |")
display(Markdown('\n'.join(lines)))
sp.to_csv(OUT + "spearman_table.csv", index=False)
print("Saved outputs/spearman_table.csv")

| System | Evaluation | n | rs | p-value |
|---|---|---|---|---|
| Fuzzy logic (original) | In-sample | 14 | 0.827 | 0.000265 |
| Fuzzy logic (redesigned) | LOOCV | 17 | 0.533 | 0.027448 |

Saved outputs/spearman_table.csv


### Notes

- Bold marks the best value per metric **among out-of-sample rows only**; in-sample
  rows are shown for contrast and are excluded from the comparison.
- `table_A_habitat.csv`, `table_B_bmwp.csv` and `table_C_numerical_bmwp.csv` are the
  direct source for Table 9 of the article.
- In-sample fuzzy rows reflect rules built from the same data used for evaluation and
  must not be read as generalisation performance.